# NOTEBOOK: 01 — Bronze Ingestion

- **Layer:** Bronze
- **Purpose:** Ingest raw Excel source files from Unity Catalog Volume and write two Delta tables preserving data exactly as received. No transformations applied.
- **Inputs:** 
    - `/Volumes/main/techmart_bronze/raw_files/electronics_dataset_products.xlsx`
    - `/Volumes/main/techmart_bronze/raw_files/electronics_dataset_vendors.xlsx`
- **Outputs:** 
    - `main.techmart_bronze.raw_products`
    - `main.techmart_bronze.raw_vendors`
- **Notes:** All columns stored as strings. Every record stamped with ingestion metadata: timestamp, source system, source file.
- **Author:** Maira Tavares


# 0. Imports

In [0]:
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
from utils.config import (
    PRODUCTS_FILE,
    VENDORS_FILE,
    BRONZE_PRODUCTS,
    BRONZE_VENDORS
)

# Ingestion metadata — stamped on every Bronze record for auditability
INGESTED_AT   = datetime.now().isoformat()
SOURCE_SYSTEM = "TechMart Vendor Feed"

print(f"  Products file : {PRODUCTS_FILE}")
print(f"  Vendors file  : {VENDORS_FILE}")
print(f"  Ingested at   : {INGESTED_AT}")

# 1. Create schemas and Volumes

In [0]:
# Create schemas
spark.sql("CREATE SCHEMA IF NOT EXISTS main.techmart_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.techmart_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.techmart_gold")

print("✅ Schemas created")
print("  main.techmart_bronze")
print("  main.techmart_silver")
print("  main.techmart_gold")

In [0]:
# Ensure Volume exists
# The Volume is the landing zone for raw source files.
# CREATE VOLUME IF NOT EXISTS is idempotent — safe to run multiple times.

spark.sql("""
    CREATE VOLUME IF NOT EXISTS main.techmart_bronze.raw_files
    COMMENT 'Raw source files for TechMart catalog pipeline'
""")

print("✅ Volume ready: /Volumes/main/techmart_bronze/raw_files/")

# 2. Reading tables

In [0]:
# Read raw Excel files
# Spark cannot read .xlsx natively — we use pandas first, then convert to Spark for Delta Lake writing.

df_products_raw = pd.read_excel(PRODUCTS_FILE, sheet_name=0)
df_vendors_raw  = pd.read_excel(VENDORS_FILE,  sheet_name=0)

print("=== Products — raw shape and columns ===")
print(f"  Rows: {len(df_products_raw)} | Columns: {df_products_raw.columns.tolist()}")

print("\n=== Vendors — raw shape and columns ===")
print(f"  Rows: {len(df_vendors_raw)} | Columns: {df_vendors_raw.columns.tolist()}")

# 3. Preparing and saving Product Table

In [0]:
# Rename columns to snake_case and cast everything to string
# Bronze = raw data, no transformations, all types kept as string
# Transformations happen in Silver

df_products_raw = df_products_raw.rename(columns={
    "ID"                  : "product_id",
    "Product Description" : "product_description_raw",
    "Weight"              : "weight_raw",
    "Unit Price"          : "unit_price_raw"
})

# Cast all to string — Bronze preserves raw values exactly as received
df_products_raw = df_products_raw.astype(str)

# Add ingestion metadata columns
df_products_raw["_ingested_at"]   = INGESTED_AT
df_products_raw["_source_system"] = SOURCE_SYSTEM
df_products_raw["_source_file"]   = PRODUCTS_FILE.split("/")[-1]

print("✅ Columns renamed and metadata added")
print("Vendors columns :", df_vendors_raw.columns.tolist())

In [0]:
# Convert to Spark DataFrames and write as Delta tables
# mergeSchema=true enables schema evolution — if new columns arrive in future feeds, the table will automatically expand to accommodate them

# Write Products Bronze table
(spark.createDataFrame(df_products_raw)
    .write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    # .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PRODUCTS))


print(f"✅ Written : {BRONZE_PRODUCTS}")
print(f"   Rows    : {spark.table(BRONZE_PRODUCTS).count()}")
display(spark.table(BRONZE_PRODUCTS))

# 4. Preparing and saving Vendors Table

In [0]:
df_vendors_raw = df_vendors_raw.rename(columns={
    "Product ID" : "product_id",
    "Vendor"     : "vendor_name_raw"
})

df_vendors_raw  = df_vendors_raw.astype(str)

df_vendors_raw["_ingested_at"]    = INGESTED_AT
df_vendors_raw["_source_system"]  = SOURCE_SYSTEM
df_vendors_raw["_source_file"]    = VENDORS_FILE.split("/")[-1]


print("✅ Columns renamed and metadata added")
print("Vendors columns :", df_vendors_raw.columns.tolist())

In [0]:
# Convert to Spark DataFrames and write as Delta tables
# mergeSchema=true enables schema evolution — if new columns arrive in future feeds, the table will automatically expand to accommodate them

# Write Vendors Bronze table
(spark.createDataFrame(df_vendors_raw).write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    # .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_VENDORS))

print(f"✅ Written : {BRONZE_VENDORS}")
print(f"   Rows    : {spark.table(BRONZE_VENDORS).count()}")
display(spark.table(BRONZE_VENDORS))